# Ghost Data Still Exists — Kaggle local experiments
No external API calls. Set the attachment paths below. Run CPU preparation before enabling a GPU. See `code/KAGGLE.md` for prerequisites and limitations.

In [ ]:
from pathlib import Path
import subprocess, sys
PROJECT = Path("/kaggle/working/project")  # Copy the source attachment here first.
DATASET = Path("/kaggle/input/CHANGE_ME/dataset")
MODEL = Path("/kaggle/input/CHANGE_ME/model")
ARTIFACTS = Path("/kaggle/working/artifacts")
assert (PROJECT / "code/main.py").is_file(), "Copy repository source into PROJECT"
assert (DATASET / "deletion_requests.csv").is_file(), "Set DATASET"
def run(*args):
    subprocess.run([sys.executable, *map(str,args)], cwd=PROJECT, check=True)


In [ ]:
run("code/main.py", "--dataset", DATASET, "--artifacts", ARTIFACTS / "sample")


Inspect extraction coverage and packet sizes before inference. Extracted OCR can still contain errors.

In [ ]:
import json
coverage = json.loads((ARTIFACTS / "sample/coverage.json").read_text())
gaps = [r for r in coverage if r["status"] != "extracted"]
print("Evidence files:", len(coverage), "Extraction gaps:", gaps)
assert not gaps, "Resolve extraction gaps before inference"


Enable a GPU for the next cell. This uses only attached weights. Quantization requires a compatible bitsandbytes installation. Model quality and memory fit must be measured.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), torch.cuda.mem_get_info())
assert (MODEL / "config.json").is_file(), "Set MODEL to attached weights"
run("code/main.py", "--dataset", DATASET, "--artifacts", ARTIFACTS / "sample", "--model-path", MODEL, "--quantize", "--output", PROJECT / "sample_predictions.csv")


In [ ]:
run("code/evaluation/main.py", "--reference", DATASET / "sample_output.csv", "--predictions", PROJECT / "sample_predictions.csv", "--report", PROJECT / "code/evaluation", "--usage", ARTIFACTS / "sample/usage.json")


Review sample errors first. Explicitly enable the full run after the experiment is acceptable. Full-run labels are hidden; do not report sample metrics as full-run accuracy.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    run("code/main.py", "--dataset", DATASET, "--requests", "deletion_requests.csv", "--artifacts", ARTIFACTS / "full", "--model-path", MODEL, "--quantize", "--output", PROJECT / "output.csv")
